In [9]:
# STEP 1: INSTALL AND IMPORT REQUIRED LIBRARIES

!python -m spacy download en_core_web_md

import spacy
import pandas as pd
from spacy.matcher import Matcher

# Load NLP model
nlp = spacy.load("en_core_web_md")

print("Libraries imported successfully!")
print("spaCy NLP model loaded successfully!")
print("NLP Pipeline:", nlp.pipe_names)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 52.9 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Libraries imported successfully!
spaCy NLP model loaded successfully!
NLP Pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [10]:
# STEP 2: CREATE CUSTOMER SUPPORT DATASET

data = {
    "Ticket": [
        "My payment failed on Amazon",
        "I was charged twice for my Flipkart order",
        "My payment was declined on Swiggy",
        "Money deducted but order not placed on Zomato",

        "Unable to login to my Amazon account",
        "My Flipkart app is not working",
        "I forgot my Swiggy account password",
        "Unable to access my Zomato account",

        "I need a refund for my Amazon order",
        "Please return my money for Flipkart order",
        "I want a refund from Swiggy",
        "My Zomato order was cancelled, need refund",

        "I want to purchase a product on Amazon",
        "Are there any discounts on Flipkart",
        "Show me the latest offers on Swiggy",
        "I want to know the pricing on Zomato"
    ],

    "Department": [
        "Billing Team", "Billing Team",
        "Billing Team", "Billing Team",
        "Technical Team", "Technical Team",
        "Technical Team", "Technical Team",
        "Refund Team", "Refund Team",
        "Refund Team", "Refund Team",
        "Sales Team", "Sales Team",
        "Sales Team", "Sales Team"
    ]
}

df = pd.DataFrame(data)

print("Dataset created successfully!")
print("Total Tickets:", len(df))

display(df)

Dataset created successfully!
Total Tickets: 16


,Ticket,Department
0,My payment failed on Amazon,Billing Team
1,I was charged twice for my Flipkart order,Billing Team
2,My payment was declined on Swiggy,Billing Team
3,Money deducted but order not placed on Zomato,Billing Team
4,Unable to login to my Amazon account,Technical Team
5,My Flipkart app is not working,Technical Team
6,I forgot my Swiggy account password,Technical Team
7,Unable to access my Zomato account,Technical Team
8,I need a refund for my Amazon order,Refund Team
9,Please return my money for Flipkart order,Refund Team


In [11]:
# STEP 3: DATASET EXPLORATION

print("Dataset Shape:", df.shape)

print("\nDataset Information:")
df.info()

print("\nDepartment Distribution:")
print(df["Department"].value_counts())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (16, 2)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Ticket      16 non-null     object
 1   Department  16 non-null     object
dtypes: object(2)
memory usage: 388.0+ bytes

Department Distribution:
Department
Billing Team      4
Technical Team    4
Refund Team       4
Sales Team        4
Name: count, dtype: int64

Missing Values:
Ticket        0
Department    0
dtype: int64


In [12]:
# STEP 4: NLP PIPELINE

def process_text(text):
    doc = nlp(text)

    tokens = [
        token.text for token in doc
        if not token.is_stop and not token.is_punct
    ]

    return " ".join(tokens)

df["Processed_Ticket"] = df["Ticket"].apply(process_text)

print("NLP Pipeline executed successfully!")

display(df[["Ticket", "Processed_Ticket"]].head(10))

NLP Pipeline executed successfully!


,Ticket,Processed_Ticket
0,My payment failed on Amazon,payment failed Amazon
1,I was charged twice for my Flipkart order,charged twice Flipkart order
2,My payment was declined on Swiggy,payment declined Swiggy
3,Money deducted but order not placed on Zomato,Money deducted order placed Zomato
4,Unable to login to my Amazon account,Unable login Amazon account
5,My Flipkart app is not working,Flipkart app working
6,I forgot my Swiggy account password,forgot Swiggy account password
7,Unable to access my Zomato account,Unable access Zomato account
8,I need a refund for my Amazon order,need refund Amazon order
9,Please return my money for Flipkart order,return money Flipkart order


In [13]:
# STEP 5: CREATE SPACY MATCHER

matcher = Matcher(nlp.vocab)

patterns = {
    "BILLING": [
        [{"LOWER": "payment"}, {"LOWER": "failed"}],
        [{"LOWER": "charged"}, {"LOWER": "twice"}],
        [{"LOWER": "payment"}, {"LOWER": "declined"}],
        [{"LOWER": "money"}, {"LOWER": "deducted"}]
    ],

    "TECHNICAL": [
        [{"LOWER": "unable"}, {"LOWER": "to"}, {"LOWER": "login"}],
        [{"LOWER": "not"}, {"LOWER": "working"}],
        [{"LOWER": "forgot"}, {"LOWER": "password"}],
        [{"LOWER": "unable"}, {"LOWER": "to"}, {"LOWER": "access"}]
    ],

    "REFUND": [
        [{"LOWER": "refund"}],
        [{"LOWER": "return"}, {"LOWER": "my"}, {"LOWER": "money"}],
        [{"LOWER": "need"}, {"LOWER": "refund"}]
    ],

    "SALES": [
        [{"LOWER": "purchase"}],
        [{"LOWER": "discounts"}],
        [{"LOWER": "offers"}],
        [{"LOWER": "pricing"}]
    ]
}

for department, pattern_list in patterns.items():
    matcher.add(department, pattern_list)

print("spaCy Matcher created successfully!")

spaCy Matcher created successfully!


In [14]:
# STEP 6: ENTITY RECOGNITION

if "entity_ruler" not in nlp.pipe_names:
    ruler = nlp.add_pipe("entity_ruler", before="ner")
else:
    ruler = nlp.get_pipe("entity_ruler")

ruler.clear()

company_patterns = [
    {"label": "COMPANY", "pattern": "Amazon"},
    {"label": "COMPANY", "pattern": "Flipkart"},
    {"label": "COMPANY", "pattern": "Swiggy"},
    {"label": "COMPANY", "pattern": "Zomato"}
]

ruler.add_patterns(company_patterns)

print("Entity Recognition configured successfully!")

Entity Recognition configured successfully!


In [15]:
# STEP 7: TEST ENTITY RECOGNITION

sample_text = [
    "My payment failed on Amazon",
    "Unable to login to Flipkart",
    "I need a refund from Swiggy",
    "I want offers from Zomato"
]

for text in sample_text:
    doc = nlp(text)

    print("\nTicket:", text)

    for ent in doc.ents:
        print("Company:", ent.text, "| Entity:", ent.label_)


Ticket: My payment failed on Amazon
Company: Amazon | Entity: COMPANY

Ticket: Unable to login to Flipkart
Company: Flipkart | Entity: COMPANY

Ticket: I need a refund from Swiggy
Company: Swiggy | Entity: COMPANY

Ticket: I want offers from Zomato
Company: Zomato | Entity: COMPANY


In [16]:
# STEP 8: TEXT SIMILARITY

department_examples = {
    "Billing Team": "payment failed billing transaction charged money deducted",
    "Technical Team": "login password account error app not working",
    "Refund Team": "refund return money reimbursement order cancellation",
    "Sales Team": "purchase buy product discount offers pricing"
}

department_docs = {
    department: nlp(description)
    for department, description in department_examples.items()
}

def find_similar_department(ticket):

    ticket_doc = nlp(ticket)

    similarity_scores = {}

    for department, department_doc in department_docs.items():

        similarity_scores[department] = ticket_doc.similarity(
            department_doc
        )

    best_department = max(
        similarity_scores,
        key=similarity_scores.get
    )

    return best_department, similarity_scores


print("Text Similarity implemented successfully!")

Text Similarity implemented successfully!


In [17]:
# STEP 9: AUTOMATIC TICKET ROUTING

department_mapping = {
    "BILLING": "Billing Team",
    "TECHNICAL": "Technical Team",
    "REFUND": "Refund Team",
    "SALES": "Sales Team"
}

def route_ticket(ticket):

    doc = nlp(ticket)
    matches = matcher(doc)

    matched_departments = {}

    for match_id, start, end in matches:

        category = nlp.vocab.strings[match_id]
        department = department_mapping[category]

        matched_departments[department] = (
            matched_departments.get(department, 0) + 1
        )

    if matched_departments:
        department = max(
            matched_departments,
            key=matched_departments.get
        )
        method = "Matcher"

    else:
        department, scores = find_similar_department(ticket)
        method = "Text Similarity"

    return department, method


print("Automatic Ticket Routing created successfully!")

Automatic Ticket Routing created successfully!


In [18]:
# STEP 10: ROUTE ALL CUSTOMER TICKETS

df["Predicted Department"] = df["Ticket"].apply(
    lambda ticket: route_ticket(ticket)[0]
)

df["Routing Method"] = df["Ticket"].apply(
    lambda ticket: route_ticket(ticket)[1]
)

print("All customer tickets routed successfully!")

display(
    df[[
        "Ticket",
        "Department",
        "Predicted Department",
        "Routing Method"
    ]]
)

All customer tickets routed successfully!


,Ticket,Department,Predicted Department,Routing Method
0,My payment failed on Amazon,Billing Team,Billing Team,Matcher
1,I was charged twice for my Flipkart order,Billing Team,Billing Team,Matcher
2,My payment was declined on Swiggy,Billing Team,Billing Team,Text Similarity
3,Money deducted but order not placed on Zomato,Billing Team,Billing Team,Matcher
4,Unable to login to my Amazon account,Technical Team,Technical Team,Matcher
5,My Flipkart app is not working,Technical Team,Technical Team,Matcher
6,I forgot my Swiggy account password,Technical Team,Technical Team,Text Similarity
7,Unable to access my Zomato account,Technical Team,Technical Team,Matcher
8,I need a refund for my Amazon order,Refund Team,Refund Team,Matcher
9,Please return my money for Flipkart order,Refund Team,Refund Team,Matcher


In [19]:
# STEP 11: EVALUATE ROUTING ACCURACY

correct_predictions = (
    df["Department"] == df["Predicted Department"]
).sum()

total_predictions = len(df)

accuracy = (correct_predictions / total_predictions) * 100

print("TICKET ROUTING EVALUATION")
print("-------------------------")
print("Total Tickets:", total_predictions)
print("Correct Predictions:", correct_predictions)
print("Incorrect Predictions:", total_predictions - correct_predictions)
print("Routing Accuracy:", round(accuracy, 2), "%")

TICKET ROUTING EVALUATION
-------------------------
Total Tickets: 16
Correct Predictions: 16
Incorrect Predictions: 0
Routing Accuracy: 100.0 %


In [20]:
# STEP 12: DEPARTMENT-WISE ROUTING RESULTS

department_results = df["Predicted Department"].value_counts()

print("DEPARTMENT-WISE CUSTOMER TICKETS")
print("--------------------------------")

display(
    department_results.to_frame(
        name="Total Tickets"
    )
)

DEPARTMENT-WISE CUSTOMER TICKETS
--------------------------------


,Total Tickets
Predicted Department,
Billing Team,4
Technical Team,4
Refund Team,4
Sales Team,4


In [21]:
# STEP 13: TEST NEW CUSTOMER TICKET

new_ticket = "My payment failed on Amazon"

department, method = route_ticket(new_ticket)

print("CUSTOMER SUPPORT TICKET")
print("-----------------------")
print("Ticket:", new_ticket)
print("Assigned Department:", department)
print("Routing Method:", method)

CUSTOMER SUPPORT TICKET
-----------------------
Ticket: My payment failed on Amazon
Assigned Department: Billing Team
Routing Method: Matcher


In [22]:
# STEP 14: TEST ALL FOUR COMPANIES

test_tickets = [
    "My payment failed on Amazon",
    "Unable to login to Flipkart",
    "I need a refund from Swiggy",
    "I want discounts on Zomato"
]

for ticket in test_tickets:

    department, method = route_ticket(ticket)

    print("Customer Ticket:", ticket)
    print("Assigned Department:", department)
    print("Routing Method:", method)
    print("-" * 50)

Customer Ticket: My payment failed on Amazon
Assigned Department: Billing Team
Routing Method: Matcher
--------------------------------------------------
Customer Ticket: Unable to login to Flipkart
Assigned Department: Technical Team
Routing Method: Matcher
--------------------------------------------------
Customer Ticket: I need a refund from Swiggy
Assigned Department: Refund Team
Routing Method: Matcher
--------------------------------------------------
Customer Ticket: I want discounts on Zomato
Assigned Department: Sales Team
Routing Method: Matcher
--------------------------------------------------


In [23]:
# STEP 15: SAVE FINAL DATASET

df.to_csv(
    "customer_support_ticket_results.csv",
    index=False
)

print("Dataset saved successfully!")
print("File: customer_support_ticket_results.csv")

Dataset saved successfully!
File: customer_support_ticket_results.csv


In [25]:
# STEP 16: DOWNLOAD DATASET

from google.colab import files

files.download("customer_support_ticket_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
# STEP 17: FINAL NLP TICKET PROCESSING

def process_ticket(ticket):

    doc = nlp(ticket)

    company = [
        ent.text for ent in doc.ents
        if ent.label_ == "COMPANY"
    ]

    department, method = route_ticket(ticket)

    return {
        "Customer Ticket": ticket,
        "Company": ", ".join(company) if company else "Not identified",
        "Assigned Department": department,
        "Routing Method": method
    }

print("Complete NLP ticket processing created successfully!")

Complete NLP ticket processing created successfully!


In [27]:
# STEP 18: FINAL TICKET ROUTING RESULTS

final_tickets = [
    "My payment failed on Amazon",
    "Unable to login to Flipkart",
    "I need a refund from Swiggy",
    "I want discounts on Zomato"
]

final_results = []

for ticket in final_tickets:
    result = process_ticket(ticket)
    final_results.append(result)

final_df = pd.DataFrame(final_results)

display(final_df)

,Customer Ticket,Company,Assigned Department,Routing Method
0,My payment failed on Amazon,Amazon,Billing Team,Matcher
1,Unable to login to Flipkart,Flipkart,Technical Team,Matcher
2,I need a refund from Swiggy,Swiggy,Refund Team,Matcher
3,I want discounts on Zomato,Zomato,Sales Team,Matcher


In [28]:
# STEP 19: FINAL PROJECT VERIFICATION

print("CUSTOMER SUPPORT TICKET ROUTING SYSTEM")
print("=" * 45)

print("\nRequired Departments:")
print(df["Department"].unique())

print("\nRecognized Companies:")
print(["Amazon", "Flipkart", "Swiggy", "Zomato"])

print("\nspaCy Concepts Implemented:")
print("1. NLP Pipeline")
print("2. Text Similarity")
print("3. Matcher")
print("4. Entity Recognition")

print("\nProject verification completed!")

CUSTOMER SUPPORT TICKET ROUTING SYSTEM

Required Departments:
['Billing Team' 'Technical Team' 'Refund Team' 'Sales Team']

Recognized Companies:
['Amazon', 'Flipkart', 'Swiggy', 'Zomato']

spaCy Concepts Implemented:
1. NLP Pipeline
2. Text Similarity
3. Matcher
4. Entity Recognition

Project verification completed!
